# 🗄️ Python Storage Formats — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Row-based storage is a filing cabinet — every drawer (row) holds everything about one person together. Fast to find a person; slow to tally everyone's salary column. Columnar storage is a spreadsheet where each column lives in its own binder — fast to SUM a column, slow to retrieve one full row. Lakehouse formats (Delta, Iceberg) add a transaction log on top of columnar files — the log is the source of truth for what's valid.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Are Storage Formats? The Visual Model](#1) |
| 2 | [Core Concepts — Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Row vs Columnar — Read Amplification](#5) |
| 6 | [Pattern 2: Parquet Encoding — Dictionary, RLE, Delta](#6) |
| 7 | [Pattern 3: Delta Lake — ACID Transactions & Time Travel](#7) |
| 8 | [Pattern 4: Apache Iceberg — Hidden Partitioning & Snapshots](#8) |
| 9 | [Pattern 5: Format Selection Matrix](#9) |
| 10 | [The Storage Formats Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Are Storage Formats? The Visual Model

---

```
ROW STORAGE (CSV, JSON, Avro):

  Disk layout: [row1_all_cols][row2_all_cols][row3_all_cols]...
  Read 1 row:  1 I/O  ← FAST (all cols of row1 are contiguous)
  Read col A:  N I/Os ← SLOW (must read every row to extract col A)
  Use case:    OLTP (transactional reads/writes of full rows)

COLUMNAR STORAGE (Parquet, ORC):

  Disk layout: [col_A: v1,v2,v3,...][col_B: v1,v2,v3,...][col_C: ...]
  Read 1 row:  N I/Os ← SLOW (each col lives in different block)
  Read col A:  1 I/O  ← FAST (all values of col A are contiguous)
  Use case:    OLAP (aggregations over large column ranges)

PARQUET INTERNAL STRUCTURE:

  File ──► Row Group (128MB default)
            ├── Column Chunk (col_A all values in this row group)
            │     ├── Page (8KB default)
            │     │     └── compressed + encoded values
            │     └── Statistics: min=10, max=9999, null_count=0
            └── Column Chunk (col_B ...)

  Footer: schema + row group metadata + column stats
  Column stats → predicate pushdown (skip entire row groups!)

DELTA LAKE TRANSACTION LOG:

  _delta_log/
    00000000000000000000.json  ← commit 0: ADD files
    00000000000000000001.json  ← commit 1: ADD + REMOVE (update)
    00000000000000000010.json  ← checkpoint (compacted log)
  Each commit = atomic operation (ACID)
  Time travel: read log at version N to get past snapshot

ICEBERG METADATA:

  catalog → current snapshot pointer
  snapshot → manifest list
  manifest list → manifest files
  manifest file → data file paths + partition info + column stats
  Hidden partitioning: partition transform stored in metadata, not in path
```


<a id='2'></a>
## 2. Core Concepts — Setup

In [ ]:
import random
import struct
import json
import hashlib
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import time

random.seed(42)

# simulate a dataset of sales records
COLUMNS = ['sale_id', 'product_id', 'region', 'amount', 'sale_date', 'quantity']
REGIONS  = ['NORTH', 'SOUTH', 'EAST', 'WEST']
PRODUCTS = [f'PROD_{i:03d}' for i in range(1, 51)]

def make_dataset(n=10000):
    rows = []
    for i in range(n):
        rows.append({
            'sale_id':    i,
            'product_id': random.choice(PRODUCTS),
            'region':     random.choice(REGIONS),
            'amount':     round(random.uniform(5.0, 500.0), 2),
            'sale_date':  f'2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}',
            'quantity':   random.randint(1, 20)
        })
    return rows

DATASET = make_dataset(10000)

# measure bytes used by row layout vs column layout
def row_size(row):
    return sum(len(str(v)) for v in row.values())

total_row_bytes = sum(row_size(r) for r in DATASET)

# columnar: each column independently
columns_data = {col: [r[col] for r in DATASET] for col in COLUMNS}
total_col_bytes = sum(len(str(v)) for col in columns_data for v in columns_data[col])

print(f"Dataset: {len(DATASET):,} rows × {len(COLUMNS)} columns")
print(f"Row storage byte estimate:    {total_row_bytes:,}")
print(f"Column storage byte estimate: {total_col_bytes:,}")
print(f"(similar raw; columnar wins via compression — same values cluster together)")
print("Setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
STORAGE FORMAT OPERATIONS
────────────────────────────────────────────────────────────────────────────
FORMAT     WRITE                READ             KEY FEATURE
────────────────────────────────────────────────────────────────────────────
CSV/JSON   append rows          parse every row  human-readable, no schema
Avro       binary row blocks    schema evolution binary row format
Parquet    columnar pages       column pruning   OLAP columnar, compress well
ORC        columnar + Bloom     pushdown         Hive native, lighter metadata
Delta Lake parquet + txn log    snapshot read    ACID, time travel, CDC
Iceberg    parquet + metadata   hidden partition schema evolution, table format
Hudi       parquet + index      upsert/merge     CDC + GDPR delete support
────────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Use CSV for production data lakes — no schema enforcement, no compression
❌  Use row format (Avro) for OLAP aggregations — column pruning impossible
❌  Use Parquet without partitioning on time-series — full scan on every query
❌  Mix Delta + Iceberg on the same table — one table format per table
❌  Write tiny files (< 128MB) in bulk — creates small file problem
❌  Skip row group statistics — required for predicate pushdown to work
```


In [ ]:
# Core API demo: column pruning and predicate pushdown simulation

# Column pruning: read only needed columns — key OLAP advantage
def read_row_format(rows, columns_needed):
    # row format: must read every column even if only 1 is needed
    bytes_read = 0
    result = []
    for row in rows:
        bytes_read += row_size(row)  # read entire row
        result.append({k: row[k] for k in columns_needed})
    return result, bytes_read

def read_columnar_format(columns_data, columns_needed, predicate=None):
    # columnar: read ONLY the requested columns — skip others entirely
    bytes_read = 0
    n = len(next(iter(columns_data.values())))
    indices = range(n)

    # predicate pushdown using column statistics (min/max per row group)
    if predicate:
        col, op, val = predicate
        col_vals = columns_data[col]
        if op == '>':
            indices = [i for i in indices if col_vals[i] > val]
        elif op == '==':
            indices = [i for i in indices if col_vals[i] == val]

    result = []
    for col in columns_needed:
        col_bytes = sum(len(str(columns_data[col][i])) for i in indices)
        bytes_read += col_bytes  # only read needed columns
    for i in indices:
        result.append({col: columns_data[col][i] for col in columns_needed})
    return result, bytes_read

print("=== Column Pruning: reading 2 of 6 columns ===")
_, rb = read_row_format(DATASET, ['amount', 'quantity'])
_, cb = read_columnar_format(columns_data, ['amount', 'quantity'])
print(f"  row format:    {rb:>10,} bytes read (read all {len(COLUMNS)} cols)")
print(f"  columnar:      {cb:>10,} bytes read (read 2/{len(COLUMNS)} cols)")
print(f"  column pruning saves: {(1 - cb/rb)*100:.0f}% of I/O")

print()
print("=== Predicate Pushdown: filter region='NORTH' ===")
_, rb2 = read_row_format([r for r in DATASET if r['region']=='NORTH'], ['amount'])
_, cb2 = read_columnar_format(columns_data, ['amount'], predicate=('region','==','NORTH'))
north_count = sum(1 for r in DATASET if r['region']=='NORTH')
print(f"  rows matching NORTH: {north_count:,} of {len(DATASET):,}")
print(f"  row format bytes:  {rb2:,} (had to read all rows to filter)")
print(f"  columnar bytes:    {cb2:,} (stats-based skip possible per row group)")

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                        WHAT TO USE
────────────────────────────────────────────────────────────────────────────
OLAP aggregations on wide tables             Parquet (columnar, compress)
OLTP row-level reads/writes                  Row format (Avro/CSV) or RDBMS
Schema evolution (add/remove columns)        Avro or Iceberg
ACID transactions on data lake               Delta Lake or Iceberg
CDC / upserts / GDPR deletes on lake         Delta Lake (MERGE) or Hudi
Time travel / audit                          Delta Lake or Iceberg
Hive/Spark ecosystem                         ORC or Parquet
Multi-cloud / multi-engine reads             Iceberg (engine-agnostic)
Streaming writes to lake                     Delta Lake (streaming ingestion)
Human-readable debug output                  JSON/CSV (never in production)
────────────────────────────────────────────────────────────────────────────
```


<a id='5'></a>
## 5. 🧩 Pattern 1: Row vs Columnar — Read Amplification

---

```
PROBLEM:
  A 1TB table has 200 columns. A BI query needs only 3 columns.
  Compare I/O cost for row vs columnar storage.

APPROACH:
  Read amplification = bytes_actually_read / bytes_of_useful_data
  Row format: amplification = 200/3 ≈ 67× for this query
  Columnar:   amplification = 1× (only the 3 columns are read)

SLOW MOTION: reading col 'amount' from 10k rows
  Row format:    read row 1 → extract amount → discard other 5 columns
                 read row 2 → extract amount → discard other 5 columns
                 ...10k rows × 6 columns = 60k field reads, 10k useful
                 I/O amplification = 6×
  Columnar:      seek to 'amount' column chunk
                 read all 10k amount values sequentially
                 I/O = exactly the amount column size
                 I/O amplification = 1×

COMPRESSION BONUS (columnar only):
  'region' column has 4 distinct values ('NORTH','SOUTH','EAST','WEST')
  Row format stores full string 10k times
  Columnar: dictionary encode → 4 entries + 10k × 2-bit index
  Compression ratio: 5 chars × 10k = 50k bytes → dict(20 bytes) + 2.5k bytes = 23× smaller

KEY INSIGHT:
  The wider the table and the fewer columns you query, the bigger columnar wins.
  For full-row OLTP reads, row format wins — no benefit to columnar.

TIME / SPACE:
  Row read (all cols):   O(cols × rows) I/O
  Columnar read (k cols): O(k × rows) I/O — independent of total column count
  Columnar compression:  typically 3–10× over row format for analytics data
```


In [ ]:
# Pattern 1: Row vs columnar read amplification

# Slow motion on 10k rows with 6 columns:
# query needs: amount, quantity (2/6 cols)
# row format:  reads 6 cols × 10k rows = 60k fields before projecting
# columnar:    reads 2 cols × 10k rows = 20k fields — exactly what's needed

def benchmark_read(label, query_cols, predicate_col=None, predicate_val=None):
    # ROW FORMAT simulation
    row_fields_read = 0
    row_result = []
    for row in DATASET:
        row_fields_read += len(COLUMNS)  # read all columns
        if predicate_col is None or row[predicate_col] == predicate_val:
            row_result.append(tuple(row[c] for c in query_cols))

    # COLUMNAR FORMAT simulation
    col_fields_read = 0
    # predicate column must be read to filter
    cols_to_read = set(query_cols)
    if predicate_col:
        cols_to_read.add(predicate_col)
    col_fields_read = len(cols_to_read) * len(DATASET)  # only needed columns

    amplification = row_fields_read / col_fields_read
    print(f"  [{label}]")
    print(f"    row format:    {row_fields_read:>8,} fields read")
    print(f"    columnar:      {col_fields_read:>8,} fields read  (amplification {amplification:.1f}×)")
    print(f"    result rows:   {len(row_result):,}")

print("=== Scenario 1: aggregation — SUM(amount) ===")
benchmark_read("SELECT SUM(amount)", ['amount'])

print()
print("=== Scenario 2: multi-column analytic query ===")
benchmark_read("SELECT region, SUM(amount), COUNT(*)", ['region', 'amount'])

print()
print("=== Scenario 3: filtered query ===")
benchmark_read("SELECT amount WHERE region='NORTH'", ['amount'],
               predicate_col='region', predicate_val='NORTH')

print()
# dictionary compression simulation
print("=== Dictionary Compression on 'region' column ===")
region_raw_bytes  = sum(len(r['region']) for r in DATASET)  # full string storage
region_dict       = sorted(set(r['region'] for r in DATASET))
region_dict_bytes = sum(len(v) for v in region_dict)  # dictionary
region_idx_bytes  = len(DATASET)  # 1 byte per index (4 values fits in 2 bits, use 1 byte)
region_enc_bytes  = region_dict_bytes + region_idx_bytes
print(f"  raw bytes:             {region_raw_bytes:,}")
print(f"  dict({len(region_dict)} entries): {region_dict_bytes} + {region_idx_bytes:,} indices = {region_enc_bytes:,} bytes")
print(f"  compression ratio:     {region_raw_bytes/region_enc_bytes:.1f}×")

print("\nRow vs columnar pattern complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Parquet Encoding — Dictionary, RLE, Delta

---

```
PROBLEM:
  A Parquet column has 1M values: 'NORTH','SOUTH','EAST','WEST' repeating.
  Show how dictionary + RLE encoding achieves 20× compression.

APPROACH:
  Three encoding layers in Parquet (applied per column per page):

  1. DICTIONARY ENCODING:
     dict = {'NORTH':0, 'SOUTH':1, 'EAST':2, 'WEST':3}
     store indices (0,1,2,3) instead of strings
     each index = 2 bits (4 values) vs 5-6 chars per string

  2. RLE (Run-Length Encoding):
     raw:    0,0,0,0,0,1,1,1,0,0,2,2,2,2...
     RLE:    (0,5),(1,3),(0,2),(2,4)...  → value + count pairs
     good for: low-cardinality columns with runs (sorted data, time-series)

  3. DELTA ENCODING (for integers/timestamps):
     raw:    100, 103, 107, 108, 115...
     deltas: first=100, +3, +4, +1, +7...
     deltas compress better than absolute values (smaller numbers)

ROW GROUP STATISTICS (min/max per column per row group):
  column 'amount': min=5.00, max=499.99
  Query: WHERE amount > 1000 → entire row group skipped — zero reads!

KEY INSIGHT:
  Parquet's compression power comes from columnar layout enabling
  dictionary + RLE on homogeneous value streams.
  Same values next to each other = better RLE = better compression.
  Sort your data before writing Parquet to maximize this.

TIME / SPACE:
  Dictionary encoding: O(N) encode/decode, storage = dict_size + N×index_bits
  RLE:                 O(N) encode, O(runs) storage — worst case O(N) if no runs
  Delta encoding:      O(N) encode/decode, storage = base + N×delta_bits
```


In [ ]:
# Pattern 2: Parquet encoding simulation

# Slow motion: encode 'region' column with dictionary + RLE
# step 1: build dict: {'NORTH':0,'SOUTH':1,'EAST':2,'WEST':3}
# step 2: convert values to indices: ['NORTH','NORTH','SOUTH',...] → [0,0,1,...]
# step 3: RLE compress indices: [(0,2),(1,1),...]

def dictionary_encode(values):
    unique = sorted(set(values))
    mapping = {v: i for i, v in enumerate(unique)}  # build dict
    indices = [mapping[v] for v in values]           # replace values with indices
    return unique, indices

def rle_encode(values):
    if not values:
        return []
    runs = []
    cur_val, cur_count = values[0], 1
    for v in values[1:]:
        if v == cur_val:
            cur_count += 1          # extend run
        else:
            runs.append((cur_val, cur_count))  # close run
            cur_val, cur_count = v, 1
    runs.append((cur_val, cur_count))  # final run
    return runs

def delta_encode(int_values):
    if not int_values:
        return [], []
    deltas = [int_values[0]]  # store first value as-is
    for i in range(1, len(int_values)):
        deltas.append(int_values[i] - int_values[i-1])  # store difference
    return deltas

region_col = [r['region'] for r in DATASET]
amount_col = sorted([int(r['amount']) for r in DATASET])  # sort for better delta

print("=== DICTIONARY ENCODING: region column ===")
dict_entries, indices = dictionary_encode(region_col)
raw_bytes  = sum(len(v) for v in region_col)
enc_bytes  = sum(len(v) for v in dict_entries) + len(indices)  # 1 byte per index (4 values)
print(f"  dictionary: {dict_entries}")
print(f"  raw bytes:  {raw_bytes:,}  encoded bytes: {enc_bytes:,}  ratio: {raw_bytes/enc_bytes:.1f}×")

print()
print("=== RLE ENCODING: sorted index stream ===")
sorted_indices = sorted(indices)  # sort to create runs
runs = rle_encode(sorted_indices)
rle_bytes = len(runs) * 2  # 2 bytes per run (value + count)
raw_index_bytes = len(indices)  # 1 byte each
print(f"  raw indices: {len(indices):,} bytes")
print(f"  RLE runs:    {len(runs)} runs → {rle_bytes} bytes")
print(f"  RLE ratio:   {raw_index_bytes/max(rle_bytes,1):.1f}×  (sorted data enables runs)")
print(f"  First 5 runs: {runs[:5]}")

print()
print("=== DELTA ENCODING: amount column (sorted integers) ===")
deltas = delta_encode(amount_col[:100])
raw_int_bytes   = 4 * 100  # 4 bytes per 32-bit int
delta_avg_bytes = sum(1 if abs(d) < 128 else 2 for d in deltas)  # variable-length
print(f"  first 5 raw values:  {amount_col[:5]}")
print(f"  first 5 deltas:      {deltas[:5]}")
print(f"  raw bytes (100 ints):   {raw_int_bytes}")
print(f"  delta bytes (var-len):  {delta_avg_bytes}")
print(f"  delta ratio: {raw_int_bytes/max(delta_avg_bytes,1):.1f}×  (small deltas fit in 1 byte)")

print()
print("=== ROW GROUP STATISTICS (predicate pushdown) ===")
# simulate row groups of 1000 rows each
rg_size = 1000
row_groups = [amount_col[i:i+rg_size] for i in range(0, len(amount_col), rg_size)]
query_min = 300  # WHERE amount > 300
skipped = sum(1 for rg in row_groups if max(rg) <= query_min)
scanned = len(row_groups) - skipped
print(f"  WHERE amount > {query_min}")
print(f"  row groups total: {len(row_groups)}, skipped: {skipped}, scanned: {scanned}")
print(f"  rows avoided: {skipped * rg_size:,} of {len(amount_col):,}")

print("\nParquet encoding pattern complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Delta Lake — ACID Transactions & Time Travel

---

```
PROBLEM:
  Two Spark jobs write to the same S3 path simultaneously.
  Without ACID, they corrupt each other's data. How does Delta solve this?

APPROACH:
  Delta Lake adds a _delta_log/ directory of JSON commit files.
  Each write = one atomic JSON commit. Concurrent writes use optimistic concurrency:
  last writer wins IFF no conflict on same files. Otherwise, retry.

SLOW MOTION: Delta commit sequence
  version 0: {"add": "part-00000.parquet"}  ← initial load
  version 1: {"add": "part-00001.parquet"}  ← new rows appended
  version 2: {"remove": "part-00000.parquet",  ← UPDATE committed
              "add":    "part-00002.parquet"}
  version 3: checkpoint → compact log into single snapshot file

TIME TRAVEL:
  SELECT * FROM events VERSION AS OF 1  ← reads version 1 snapshot
  SELECT * FROM events TIMESTAMP AS OF '2024-03-01'  ← time-based
  Implementation: replay log from 0 to target version → file list

ACID GUARANTEES:
  Atomicity:   commit file written atomically (JSON append)
  Consistency: schema enforced on write (rejects bad columns)
  Isolation:   snapshot isolation — readers see consistent snapshot
  Durability:  commit file persisted before data files referenced

KEY INSIGHT:
  Delta's transaction log IS the source of truth.
  Data files exist but are "invisible" until referenced in a committed log entry.
  VACUUM removes files no longer referenced by any live snapshot.

TIME / SPACE:
  Commit write:  O(1) — append one JSON line
  Time travel:   O(V) — replay V versions of log
  Checkpoint:    O(N) — compact N log entries into one
  VACUUM:        O(F) — scan file references, delete unreferenced
```


In [ ]:
# Pattern 3: Delta Lake transaction log simulation

# Slow motion: 4-commit sequence showing ACID properties
# commit 0: initial load — ADD part-00000
# commit 1: append        — ADD part-00001
# commit 2: update        — REMOVE part-00000, ADD part-00002 (new version of file)
# commit 3: delete        — REMOVE part-00001 (records meeting delete predicate)

from dataclasses import dataclass
from typing import Optional

@dataclass
class DeltaAction:
    action: str  # 'add' or 'remove'
    path:   str
    size_bytes: int = 0
    stats: Optional[dict] = None

@dataclass
class DeltaCommit:
    version:   int
    timestamp: str
    operation: str
    actions:   List[DeltaAction]

class DeltaLog:
    """
    Storage Format Pattern 3 — Delta Lake transaction log simulation.
    Approach: Append-only log of commit entries; snapshot at any version by replaying.
    Time:  O(V) for time travel to version V
    Space: O(N) for N total committed files (including removed)
    """
    def __init__(self):
        self.log: List[DeltaCommit] = []
        self.version = -1

    def commit(self, operation: str, adds=None, removes=None):
        self.version += 1
        actions = []
        for path, size in (adds or []):
            actions.append(DeltaAction('add', path, size))
        for path in (removes or []):
            actions.append(DeltaAction('remove', path))
        entry = DeltaCommit(self.version, f'2024-01-{self.version+1:02d}', operation, actions)
        self.log.append(entry)
        return self.version

    def snapshot_at(self, version: int) -> set:
        # replay log from 0 to version → set of live files
        live_files = set()
        for commit in self.log:
            if commit.version > version:
                break
            for action in commit.actions:
                if action.action == 'add':
                    live_files.add(action.path)    # file becomes visible
                elif action.action == 'remove':
                    live_files.discard(action.path)  # file hidden from queries
        return live_files

    def vacuum(self, retain_versions=2) -> List[str]:
        # find files safe to delete — not referenced by any snapshot >= (current - retain)
        safe_version = max(0, self.version - retain_versions)
        all_added   = {a.path for c in self.log for a in c.actions if a.action == 'add'}
        still_live  = self.snapshot_at(self.version)  # current live files
        older_live  = set()
        for v in range(safe_version, self.version + 1):
            older_live |= self.snapshot_at(v)  # files needed for time travel
        deletable = all_added - older_live
        return list(deletable)

    def print_log(self):
        for commit in self.log:
            adds    = [a.path for a in commit.actions if a.action == 'add']
            removes = [a.path for a in commit.actions if a.action == 'remove']
            print(f"  v{commit.version} [{commit.operation}]: +{adds} -{removes}")

delta = DeltaLog()

v0 = delta.commit('CREATE', adds=[('part-000.parquet', 50000), ('part-001.parquet', 48000)])
v1 = delta.commit('APPEND', adds=[('part-002.parquet', 52000)])
v2 = delta.commit('UPDATE', adds=[('part-003.parquet', 49000)], removes=['part-000.parquet'])
v3 = delta.commit('DELETE', removes=['part-001.parquet'])

print("=== Delta Log ===")
delta.print_log()

print()
print("=== Time Travel ===")
for v in range(4):
    snap = delta.snapshot_at(v)
    print(f"  VERSION {v}: {sorted(snap)}")

print()
print("=== VACUUM (retain 2 versions) ===")
deletable = delta.vacuum(retain_versions=2)
print(f"  Safe to delete: {deletable}")
print(f"  (files removed from log but still kept for time travel within retention window)")

print("\nDelta Lake pattern complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Apache Iceberg — Hidden Partitioning & Snapshots

---

```
PROBLEM:
  A Delta Lake table was partitioned by YEAR(sale_date). Queries using
  WHERE sale_date = '2024-03-15' miss the partition because the physical
  path is year=2024/, not date=2024-03-15/. Users must know the partition scheme.
  How does Iceberg solve this?

APPROACH:
  Iceberg uses hidden partitioning: the partition transform is stored in
  TABLE METADATA, not in the physical file path. Queries don't need to
  know the partition scheme — Iceberg applies the transform automatically.

ICEBERG METADATA HIERARCHY:
  catalog.table → current_snapshot_id → snapshot
  snapshot → manifest_list.avro → [manifest_1.avro, manifest_2.avro]
  manifest → [file_path, partition_value, row_count, col_stats]

SLOW MOTION: hidden partition write
  Row: {sale_date: '2024-03-15', amount: 99.99}
  Partition spec: transform(sale_date, 'month') → '2024-03'
  File written to: data/sale_month=2024-03/part-00001.parquet
  Query: WHERE sale_date = '2024-03-15'
  Iceberg: apply month() to '2024-03-15' → '2024-03' → prune all other months
  User sees: no partition syntax needed in query

SCHEMA EVOLUTION:
  Iceberg tracks column IDs (not names) — renaming is safe
  Add column:    safe (nullable, default=null for old files)
  Drop column:   safe (old files still have data, new reads project null)
  Rename column: safe (ID unchanged, name update in metadata only)
  Reorder:       safe (physical order irrelevant — ID-based mapping)

KEY INSIGHT:
  Iceberg never modifies data files — all changes are metadata-only.
  This makes schema evolution, partition evolution, and rollback O(1) operations.

TIME / SPACE:
  Write:           O(1) metadata update + O(rows) data file write
  Read (pruned):   O(manifests_in_snapshot) to find files + O(matching_rows)
  Schema change:   O(1) — metadata update only
  Snapshot rollback: O(1) — update current_snapshot pointer
```


In [ ]:
# Pattern 4: Iceberg hidden partitioning simulation

# Slow motion: write events with month partition transform
# step 1: row arrives with sale_date='2024-03-15'
# step 2: apply month() transform → partition_value = '2024-03'
# step 3: route to manifest for '2024-03' partition
# step 4: query WHERE sale_date='2024-03-15' → apply month() → prune to '2024-03'

@dataclass
class DataFile:
    path:            str
    partition_value: str
    row_count:       int
    col_stats:       dict  # min/max per column

@dataclass
class Snapshot:
    snapshot_id: int
    files:       List[DataFile]
    parent_id:   Optional[int] = None

class IcebergTable:
    """
    Storage Format Pattern 4 — Iceberg metadata and hidden partitioning.
    Approach: Partition transform in metadata; query engine applies transform transparently.
    Time:  O(F) to scan manifests, O(matching_files × rows_per_file) to scan data
    Space: O(snapshots × files) for metadata
    """
    def __init__(self, name: str, partition_col: str, transform: str):
        self.name = name
        self.partition_col = partition_col
        self.transform = transform  # 'month', 'year', 'bucket', 'identity'
        self.snapshots: List[Snapshot] = []
        self.current_snapshot_id: Optional[int] = None
        self._file_counter = 0

    def _apply_transform(self, value: str) -> str:
        if self.transform == 'month':
            return value[:7]  # '2024-03-15' → '2024-03'
        elif self.transform == 'year':
            return value[:4]  # '2024-03-15' → '2024'
        return value  # identity

    def write(self, rows):
        # group rows by partition value
        by_partition = defaultdict(list)
        for row in rows:
            pval = self._apply_transform(row[self.partition_col])
            by_partition[pval].append(row)
        # create data files
        new_files = []
        for pval, prows in by_partition.items():
            self._file_counter += 1
            amounts = [r['amount'] for r in prows]
            f = DataFile(
                path=f'data/{self.partition_col}_month={pval}/part-{self._file_counter:05d}.parquet',
                partition_value=pval,
                row_count=len(prows),
                col_stats={'amount': {'min': min(amounts), 'max': max(amounts)}}
            )
            new_files.append(f)
        # inherit previous snapshot files + add new
        prev_files = []
        if self.current_snapshot_id is not None:
            prev = next(s for s in self.snapshots if s.snapshot_id == self.current_snapshot_id)
            prev_files = prev.files[:]
        snap_id = len(self.snapshots)
        snap = Snapshot(snap_id, prev_files + new_files, self.current_snapshot_id)
        self.snapshots.append(snap)
        self.current_snapshot_id = snap_id
        return snap_id

    def query(self, date_predicate: str, amount_min: float = None):
        # hidden partitioning: apply transform to predicate transparently
        pred_pval = self._apply_transform(date_predicate)
        snap = next(s for s in self.snapshots if s.snapshot_id == self.current_snapshot_id)
        pruned = []
        for f in snap.files:
            if f.partition_value != pred_pval:
                continue  # partition pruned — zero rows read
            if amount_min and f.col_stats['amount']['max'] < amount_min:
                continue  # column stats pruning — skip entire file
            pruned.append(f)
        total_possible = sum(f.row_count for f in snap.files)
        rows_scanned   = sum(f.row_count for f in pruned)
        print(f"    predicate date='{date_predicate}' → partition='{pred_pval}'")
        print(f"    files pruned: {len(snap.files) - len(pruned)}/{len(snap.files)} skipped")
        print(f"    rows scanned: {rows_scanned:,}/{total_possible:,}")

    def rollback(self, snapshot_id: int):
        self.current_snapshot_id = snapshot_id  # O(1) metadata pointer update
        print(f"    rolled back to snapshot {snapshot_id}")

tbl = IcebergTable('sales', 'sale_date', 'month')
# write 3 batches: Jan, Feb, Mar data
s0 = tbl.write([r for r in DATASET if r['sale_date'][:7] in ['2024-01', '2024-02']][:300])
s1 = tbl.write([r for r in DATASET if r['sale_date'][:7] == '2024-03'][:200])

print("=== Current snapshot files ===")
snap = tbl.snapshots[tbl.current_snapshot_id]
for f in snap.files:
    print(f"  {f.partition_value}: {f.row_count} rows  {f.path.split('/')[-1]}")

print()
print("=== Hidden partition query (no partition syntax needed) ===")
tbl.query('2024-03-15')

print()
print("=== Column stats pushdown: WHERE amount > 400 ===")
tbl.query('2024-03-15', amount_min=400)

print()
print("=== Time travel: rollback to snapshot 0 ===")
tbl.rollback(s0)
snap_old = tbl.snapshots[s0]
print(f"    files in v0 snapshot: {len(snap_old.files)}")

print("\nIceberg pattern complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Format Selection Matrix

---

```
PROBLEM:
  You're designing a new data lake. Which format should you choose for:
  a) raw ingestion zone  b) curated OLAP tables  c) ML feature store  d) CDC pipeline

DECISION FRAMEWORK:

  QUESTION 1: Do you need ACID transactions?
    No  → Parquet (simple, widely supported)
    Yes → Delta Lake or Iceberg

  QUESTION 2: Do you need schema evolution or partition evolution?
    Simple add/drop columns → Parquet is OK
    Complex evolution, rename → Iceberg (ID-based columns)

  QUESTION 3: Do you need time travel / audit?
    Yes → Delta Lake (easier syntax) or Iceberg (more flexible)

  QUESTION 4: What engine reads this?
    Spark/Databricks primarily → Delta Lake (native)
    Multi-engine (Trino, Flink, Spark, Snowflake) → Iceberg (open standard)
    Hive → ORC

  QUESTION 5: GDPR deletes or CDC (upserts)?
    Yes → Delta MERGE or Hudi (merge-on-read for high-churn CDC)

FORMAT COMPARISON:
               CSV    Avro   Parquet  ORC    Delta    Iceberg  Hudi
  Columnar     No     No     Yes      Yes    Yes      Yes      Yes
  Compression  Poor   Good   Best     Best   Best     Best     Best
  ACID         No     No     No       No     Yes      Yes      Yes
  Time travel  No     No     No       No     Yes      Yes      Partial
  Schema evol  No     Best   Limited  Good   Good     Best     Good
  Streaming    Yes    Yes    Limited  No     Yes      Yes      Yes
  GDPR delete  Full   Full   Hard     Hard   MERGE    Yes      Best

KEY INSIGHT:
  Default pick for new lakehouse: Iceberg (open, multi-engine, best schema evolution)
  Default pick for Databricks-only shop: Delta Lake (native, easy, well-documented)
  High-churn CDC with frequent updates: Hudi (merge-on-read optimized)

TIME / SPACE:
  Parquet compression ratio: 3–10× vs CSV (analytics workloads)
  Delta/Iceberg overhead:    ~1-5% for metadata vs raw Parquet
  Small file merge (OPTIMIZE): reduces N small files to M large — O(N × rows)
```


In [ ]:
# Pattern 5: Format selection logic

# Slow motion: decision tree for format selection
# step 1: check ACID requirement (yes → Delta/Iceberg, no → Parquet)
# step 2: check engine ecosystem (Databricks → Delta, multi-engine → Iceberg)
# step 3: check schema evolution needs (complex → Iceberg, simple → either)
# step 4: check CDC/GDPR (high churn → Hudi, occasional → Delta MERGE)

def select_format(acid_needed, time_travel, engine, schema_evolution, cdc_gdpr):
    """
    Format selection decision tree.
    Args:
        acid_needed (bool):     concurrent writes or strict consistency
        time_travel (bool):     audit / rollback requirements
        engine (str):           'databricks', 'multi-engine', 'hive', 'any'
        schema_evolution (str): 'simple', 'complex' (renames, reorders)
        cdc_gdpr (bool):        frequent upserts or GDPR delete required
    Returns:
        str: recommended format with rationale
    """
    if not acid_needed and not time_travel and engine == 'hive':
        return 'ORC', 'Hive native, columnar, good compression'
    if not acid_needed and not time_travel:
        return 'Parquet', 'Simplest, most compatible, best compression'
    if cdc_gdpr:
        return 'Hudi', 'Merge-on-read optimized for high-churn CDC + GDPR'
    if acid_needed and engine == 'databricks':
        return 'Delta Lake', 'Native Databricks, best tooling, ACID + time travel'
    if acid_needed and schema_evolution == 'complex':
        return 'Iceberg', 'ID-based columns = safe rename/reorder, multi-engine'
    if acid_needed:
        return 'Iceberg', 'Open standard, multi-engine ACID, growing ecosystem'
    return 'Parquet', 'Default: no ACID requirements'

scenarios = [
    ("Raw landing zone (S3)",
     dict(acid_needed=False, time_travel=False, engine='any', schema_evolution='simple', cdc_gdpr=False)),
    ("Curated OLAP (Databricks)",
     dict(acid_needed=True,  time_travel=True,  engine='databricks', schema_evolution='simple', cdc_gdpr=False)),
    ("Multi-engine lakehouse (Spark+Trino)",
     dict(acid_needed=True,  time_travel=True,  engine='multi-engine', schema_evolution='complex', cdc_gdpr=False)),
    ("CDC pipeline (frequent upserts)",
     dict(acid_needed=True,  time_travel=False, engine='any', schema_evolution='simple', cdc_gdpr=True)),
    ("Hive DW (legacy)",
     dict(acid_needed=False, time_travel=False, engine='hive', schema_evolution='simple', cdc_gdpr=False)),
    ("GDPR-sensitive user data",
     dict(acid_needed=True,  time_travel=False, engine='any', schema_evolution='simple', cdc_gdpr=True)),
]

print("=== FORMAT SELECTION MATRIX ===")
for label, params in scenarios:
    fmt, reason = select_format(**params)
    print(f"  {label:40s} → {fmt:12s} ({reason})")

print()
# compression ratio simulation
print("=== COMPRESSION COMPARISON SIMULATION ===")
csv_bytes  = sum(len(','.join(str(v) for v in r.values())) for r in DATASET[:1000])
# simulate parquet: dictionary + RLE compression ~ 5× over CSV for this data
parquet_bytes = csv_bytes // 5
delta_bytes   = parquet_bytes + 512  # delta log overhead per batch
print(f"  CSV (1k rows):     {csv_bytes:,} bytes")
print(f"  Parquet (est):     {parquet_bytes:,} bytes  ({csv_bytes/parquet_bytes:.1f}× smaller)")
print(f"  Delta Lake (est):  {delta_bytes:,} bytes  (parquet + ~512B log overhead per commit)")

print("\nFormat selection pattern complete.")

<a id='10'></a>
## 10. The Storage Formats Decision Map

---

```
QUESTION                            FORMAT       REASON
────────────────────────────────────────────────────────────────────────────
No schema, human-readable           CSV          debug only, never production
Schema evolution (Kafka source)     Avro         row format, schema registry
OLAP analytics (Spark/Athena)       Parquet      columnar, compress, pushdown
OLAP analytics (Hive)               ORC          Hive native, similar to Parquet
ACID + time travel (Databricks)     Delta Lake   native, easiest, CDC support
ACID + multi-engine (Trino+Spark)   Iceberg      open standard, schema evolution
Frequent CDC / upserts              Hudi         merge-on-read, GDPR delete
────────────────────────────────────────────────────────────────────────────

PARQUET INTERNALS CHEAT SHEET:
  Row group:     128MB (default) — unit of parallelism
  Page size:     8KB — smallest unit of compression
  Encoding:      PLAIN, DICTIONARY, RLE, DELTA, BYTE_STREAM_SPLIT
  Compression:   SNAPPY (fast), ZSTD (best ratio), GZIP (legacy)
  Statistics:    min/max/null_count per column per row group
  Column pruning: read only requested columns
  Row group skip: stats-based predicate pushdown

DELTA VS ICEBERG:
  Delta Lake:  Databricks-native, great tooling, DML (MERGE/UPDATE/DELETE)
  Iceberg:     Engine-agnostic (Spark/Flink/Trino/Snowflake), hidden partitioning,
               better schema evolution (column IDs), partition evolution
  Choose Delta: Databricks shop, need fastest onboarding
  Choose Iceberg: multi-engine, long-lived table, schema churn expected
```


<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for each format:

| Signal | What to Use |
|--------|-------------|
| "OLAP query on wide table" | Parquet — column pruning |
| "Need transactions on lake" | Delta Lake or Iceberg |
| "Multi-engine reads" | Iceberg — open standard |
| "High-churn CDC" | Hudi or Delta MERGE |
| "Schema may change" | Iceberg (ID-based) |
| "Time travel / audit" | Delta or Iceberg |
| "Hive ecosystem" | ORC |

---

### Key numbers — memorize these:

```
Parquet row group size:   128MB (default)
Parquet page size:        8KB
Parquet compression:      3-10× vs CSV for analytics data
Delta log checkpoint:     every 10 commits (default)
Iceberg snapshot pointer: 1 metadata.json file update per commit
Small file threshold:     < 128MB → triggers compaction (OPTIMIZE)
```

---

### Common templates:

```python
# TEMPLATE: Parquet write with partitioning (PySpark)
df.write.format('parquet') \
    .partitionBy('year', 'month') \
    .option('compression', 'snappy') \
    .mode('overwrite') \
    .save('s3://bucket/table/')

# TEMPLATE: Delta MERGE (upsert)
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, 's3://bucket/table/')
dt.alias('t').merge(
    updates.alias('u'),
    't.id = u.id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# TEMPLATE: Delta time travel
df = spark.read.format('delta').option('versionAsOf', 5).load('s3://...')

# TEMPLATE: Iceberg partition evolution (no rewrite)
# ALTER TABLE t REPLACE PARTITION FIELD date WITH month(date)
```

---

### Gotchas to not forget:

```
❌  Writing unsorted data to Parquet — poor RLE, worse stats, missed pushdown
❌  Too many partitions (1000+) → metadata explosion → slow planning
❌  Too few partitions → large files, poor parallelism
❌  Mixing row group sizes — some tools expect consistent 128MB
✅  Sort by high-cardinality column before write → better RLE + smaller pages
✅  VACUUM Delta regularly — unreferenced files waste storage
✅  Z-ORDER in Delta = multi-column sort → better predicate pushdown
✅  Iceberg hidden partitioning = no partition filter needed in queries
```


<a id='12'></a>
## 12. Summary Map

---

```
                     🗄️ STORAGE FORMATS
                              │
        ┌─────────────────────┼──────────────────────┐
        │                     │                      │
  ROW FORMATS          COLUMNAR               LAKEHOUSE
  (Pattern 1)          (Parquet/ORC)          (Delta/Iceberg/Hudi)
        │              (Patterns 1,2)         (Patterns 3,4,5)
  CSV/JSON/Avro              │                      │
  OLTP / ingest        Dictionary encode    Transaction Log (Delta)
  Schema evolution     RLE compression      Metadata tree (Iceberg)
  (Avro)               Delta encoding       Merge-on-read (Hudi)
                       Row group stats      ACID, Time Travel
                       Column pruning       Schema/partition evolution

SELECTION FLOWCHART:
  Need ACID?
    No  → Parquet (OLAP) / Avro (streaming)
    Yes → Databricks shop? → Delta Lake
         Multi-engine?     → Iceberg
         High-churn CDC?   → Hudi

FORMAT COMPRESSION LADDER (best to worst for analytics):
  ZSTD Parquet > Snappy Parquet > ORC Zlib > Avro Snappy > CSV (uncompressed)
```

---
*End of Storage Formats Master Guide — Sean Edition*
